## Telecom Dataset Generator

Generates a synthetic **telecom** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `telecom` | `subscribers` | ~5K | `call_records` | 100K-500K | Temporal churn dates, network quality metrics (latency/signal), region-correlated towers |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `telecom` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.telecom') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.telecom');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(55)
random.seed(55)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "telecom"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = datetime(2026, 3, 21)

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Subscribers table (~5,000 rows) ---
regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West Coast", "Pacific Northwest", "Mountain West", "Mid-Atlantic"]

plan_profiles = {
    "Basic":     {"monthly_charge": 29.99, "data_limit_gb": 2,
                  "contract_weights": [50, 10, 5, 35],
                  "device_weights": [0, 5, 0, 10, 0, 15, 5, 25, 30, 10],
                  "5g_prob": 0.15},
    "Standard":  {"monthly_charge": 49.99, "data_limit_gb": 10,
                  "contract_weights": [30, 35, 20, 15],
                  "device_weights": [5, 15, 10, 20, 10, 15, 10, 10, 0, 5],
                  "5g_prob": 0.45},
    "Premium":   {"monthly_charge": 79.99, "data_limit_gb": 50,
                  "contract_weights": [15, 25, 50, 10],
                  "device_weights": [25, 15, 25, 10, 15, 5, 5, 0, 0, 0],
                  "5g_prob": 0.80},
    "Unlimited": {"monthly_charge": 99.99, "data_limit_gb": -1,
                  "contract_weights": [10, 20, 60, 10],
                  "device_weights": [30, 10, 30, 5, 10, 5, 5, 0, 5, 0],
                  "5g_prob": 0.85},
    "Family":    {"monthly_charge": 129.99, "data_limit_gb": 25,
                  "contract_weights": [5, 20, 65, 10],
                  "device_weights": [15, 15, 15, 15, 10, 10, 5, 10, 5, 0],
                  "5g_prob": 0.60},
}
plans = list(plan_profiles.keys())
device_types = ["iPhone 15", "iPhone 14", "Samsung Galaxy S24", "Samsung Galaxy S23", "Google Pixel 8",
                "Google Pixel 7", "OnePlus 12", "Motorola Edge", "iPhone SE", "Samsung Galaxy A54"]
contract_types = ["Monthly", "12-Month", "24-Month", "Prepaid"]
statuses = ["Active", "Suspended", "Churned", "Pending Activation"]

# region_towers is built with the towers dimension table (after subscribers)

NUM_SUBSCRIBERS = 5000
subscribers = []
for i in range(1, NUM_SUBSCRIBERS + 1):
    activation = date(2018, 1, 1) + timedelta(days=random.randint(0, 2800))
    tenure_years = (NOW.date() - activation).days / 365.25
    plan = random.choices(plans, weights=[20, 30, 20, 15, 15])[0]
    profile = plan_profiles[plan]

    contract = random.choices(contract_types, weights=profile["contract_weights"])[0]
    device = random.choices(device_types, weights=profile["device_weights"])[0]
    is_5g = random.random() < profile["5g_prob"]
    credit = int(clamp(random.gauss(700, 80), 300, 850))
    region = random.choice(regions)

    # Churn: temporal event with a date (not just a flag)
    churn_base = 0.12 if plan in ["Basic", "Standard"] else 0.06
    churn_prob = churn_base * (1.0 / max(0.5, tenure_years))
    if random.random() < churn_prob:
        status = "Churned"
        # Churn happened sometime between activation and now
        churn_offset = random.randint(int(tenure_years * 180), max(int(tenure_years * 365), 365))
        churn_date = activation + timedelta(days=min(churn_offset, (NOW.date() - activation).days))
    elif random.random() < 0.05:
        status = "Suspended"
        churn_date = None
    elif random.random() < 0.03:
        status = "Pending Activation"
        churn_date = None
    else:
        status = "Active"
        churn_date = None

    # Monthly data usage: correlated with plan
    if plan == "Unlimited":
        avg_monthly_gb = round(clamp(random.lognormvariate(3.2, 0.6), 5.0, 150.0), 1)
    elif plan == "Premium":
        avg_monthly_gb = round(clamp(random.lognormvariate(2.8, 0.5), 3.0, 80.0), 1)
    elif plan == "Family":
        avg_monthly_gb = round(clamp(random.lognormvariate(2.5, 0.6), 2.0, 60.0), 1)
    elif plan == "Standard":
        avg_monthly_gb = round(clamp(random.lognormvariate(2.0, 0.5), 1.0, 30.0), 1)
    else:
        avg_monthly_gb = round(clamp(random.lognormvariate(1.2, 0.5), 0.5, 10.0), 1)

    subscribers.append(Row(
        subscriber_id=i,
        subscriber_name=fake.name(),
        phone_number=fake.phone_number(),
        plan_name=plan,
        contract_type=contract,
        monthly_charge=profile["monthly_charge"],
        data_limit_gb=profile["data_limit_gb"],
        device_type=device,
        region=region,
        status=status,
        activation_date=activation,
        churn_date=churn_date,
        credit_score=credit,
        is_5g_enabled=is_5g,
        avg_monthly_data_gb=avg_monthly_gb
    ))

sub_lookup = {s.subscriber_id: s for s in subscribers}

subscribers_df = spark.createDataFrame(subscribers)
subscribers_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.subscribers")
print(f"✔ Created {CATALOG_SCHEMA}.subscribers ({subscribers_df.count()} rows)")

# --- Towers dimension table (formalizes region tower pools) ---
tower_technologies = ["5G", "4G LTE", "3G", "Wi-Fi"]
tower_tech_weights = [30, 45, 15, 10]
seen_tower_ids = set()
towers = []
region_towers = {r: [] for r in regions}
for r in regions:
    prefix = r[:2].upper()
    for _ in range(25):
        while True:
            tid = f"TWR-{prefix}-{random.randint(100, 999)}"
            if tid not in seen_tower_ids:
                seen_tower_ids.add(tid)
                break
        region_towers[r].append(tid)
        cap = int(round(clamp(random.lognormvariate(math.log(2200), 0.35), 500, 5000)))
        towers.append(Row(
            tower_id=tid,
            region=r,
            latitude=round(random.uniform(25.0, 48.0), 6),
            longitude=round(random.uniform(-125.0, -70.0), 6),
            technology=random.choices(tower_technologies, weights=tower_tech_weights)[0],
            capacity_subscribers=cap,
            sector_count=random.randint(1, 6),
        ))

towers_df = spark.createDataFrame(towers)
towers_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.towers")
print(f"✔ Created {CATALOG_SCHEMA}.towers ({towers_df.count()} rows)")

# --- Call/Usage Records table (randomized ~100K-500K rows) ---
record_types = ["Voice Call", "SMS", "Data Session", "MMS", "Voicemail"]
call_results = ["Completed", "Missed", "Dropped", "Busy", "Voicemail"]
roaming_countries = [None, None, None, None, None, None, "Canada", "Mexico", "UK", "Germany", "Japan"]

plan_record_weights = {
    "Basic":     [40, 30, 12, 5, 13],
    "Standard":  [35, 25, 25, 5, 10],
    "Premium":   [25, 15, 45, 5, 10],
    "Unlimited": [20, 12, 55, 3, 10],
    "Family":    [30, 20, 35, 5, 10],
}

records = []
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
n = 0
while n < NUM_EVENT_RECORDS:
    record_ts = datetime(2024, 6, 1) + timedelta(days=random.randint(0, 600), hours=random.randint(0, 23), minutes=random.randint(0, 59))

    sid = random.randint(1, NUM_SUBSCRIBERS)
    sub = sub_lookup[sid]
    if sub.status == "Churned" and sub.churn_date is not None and record_ts.date() > sub.churn_date:
        continue

    n += 1
    rec_type = random.choices(record_types, weights=plan_record_weights.get(sub.plan_name, [30,20,30,5,15]))[0]

    if rec_type == "Voice Call":
        duration = round(clamp(random.lognormvariate(1.5, 0.9), 0.1, 180.0), 1)
    elif rec_type in ["SMS", "MMS"]:
        duration = 0.0
    else:
        duration = round(clamp(random.lognormvariate(2.5, 1.2), 0.5, 480.0), 1)

    if rec_type == "Data Session":
        if sub.plan_name in ["Unlimited", "Premium"]:
            data_mb = round(clamp(random.lognormvariate(4.5, 1.5), 0.1, 8192.0), 2)
        elif sub.plan_name == "Standard":
            data_mb = round(clamp(random.lognormvariate(3.5, 1.5), 0.01, 4096.0), 2)
        else:
            data_mb = round(clamp(random.lognormvariate(2.5, 1.5), 0.01, 2048.0), 2)
    else:
        data_mb = round(clamp(random.expovariate(1.0), 0, 5.0), 2)

    if sub.is_5g_enabled:
        network = random.choices(["5G", "4G LTE", "3G", "Wi-Fi Calling"], weights=[55, 30, 2, 13])[0]
    else:
        network = random.choices(["5G", "4G LTE", "3G", "Wi-Fi Calling"], weights=[0, 65, 15, 20])[0]

    roaming = random.choice(roaming_countries)
    if roaming is not None:
        charge = round(clamp(random.lognormvariate(1.5, 0.8), 0.50, 50.0), 2)
    elif sub.data_limit_gb > 0 and rec_type == "Data Session" and data_mb > sub.data_limit_gb * 1024 * 0.8:
        charge = round(clamp(random.lognormvariate(1.0, 0.6), 0.25, 25.0), 2)
    else:
        charge = 0.0

    # Network quality metrics: correlated with network type and region
    if network == "5G":
        latency = round(clamp(random.gauss(12, 4), 3.0, 40.0), 1)
        signal = round(clamp(random.gauss(-75, 10), -110, -50), 0)
    elif network == "4G LTE":
        latency = round(clamp(random.gauss(30, 10), 10.0, 80.0), 1)
        signal = round(clamp(random.gauss(-85, 12), -120, -60), 0)
    elif network == "3G":
        latency = round(clamp(random.gauss(80, 25), 30.0, 200.0), 1)
        signal = round(clamp(random.gauss(-95, 10), -120, -70), 0)
    else:
        latency = round(clamp(random.gauss(20, 8), 5.0, 60.0), 1)
        signal = round(clamp(random.gauss(-65, 8), -90, -40), 0)

    # Use region-correlated towers
    tower = random.choice(region_towers.get(sub.region, [f"TWR-XX-{random.randint(100,999)}"]))

    records.append(Row(
        record_id=10000 + n,
        subscriber_id=sid,
        record_date=record_ts,
        record_type=rec_type,
        duration_minutes=duration,
        data_usage_mb=data_mb,
        charge_amount=charge,
        network_type=network,
        call_result=random.choices(call_results, weights=[70,12,8,5,5])[0] if rec_type == "Voice Call" else "Completed",
        tower_id=tower,
        roaming_country=roaming,
        is_international=roaming is not None,
        latency_ms=latency,
        signal_strength_dbm=int(signal)
    ))

records_df = spark.createDataFrame(records)
records_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.call_records")
print(f"✔ Created {CATALOG_SCHEMA}.call_records ({records_df.count()} rows)")

# --- Billing invoices (monthly, Active/Churned; capped ~30K–60K rows) ---
MAX_INVOICES = 55_000
_bill_rng = random.Random(55)
billing_candidates = []
for s in subscribers:
    if s.status not in ("Active", "Churned"):
        continue
    if s.status == "Churned" and s.churn_date is None:
        continue
    end_m = date(s.churn_date.year, s.churn_date.month, 1) if (s.status == "Churned" and s.churn_date) else date(2025, 12, 1)
    cur = date(s.activation_date.year, s.activation_date.month, 1)
    if end_m < cur:
        continue
    while cur <= end_m:
        billing_candidates.append((s.subscriber_id, cur, s))
        if cur.month == 12:
            cur = date(cur.year + 1, 1, 1)
        else:
            cur = date(cur.year, cur.month + 1, 1)

if len(billing_candidates) > MAX_INVOICES:
    billing_candidates = _bill_rng.sample(billing_candidates, MAX_INVOICES)
billing_candidates.sort(key=lambda x: (x[0], x[1]))

payment_statuses = ["Paid", "Pending", "Overdue", "Failed"]
payment_weights = [88, 5, 5, 2]
invoices = []
for invoice_id, (sid, billing_month, s) in enumerate(billing_candidates, start=1):
    profile = plan_profiles[s.plan_name]
    base_charge = profile["monthly_charge"]
    if profile["data_limit_gb"] < 0:
        usage_charges = 0.0
    else:
        usage_charges = round(clamp(random.lognormvariate(math.log(3), 0.7), 0.0, 45.0), 2)
    total_charge = round(base_charge + usage_charges, 2)
    payment_status = random.choices(payment_statuses, weights=payment_weights)[0]
    if payment_status == "Paid":
        payment_date = billing_month + timedelta(days=random.randint(1, 30))
    else:
        payment_date = None
    invoices.append(Row(
        invoice_id=invoice_id,
        subscriber_id=sid,
        billing_month=billing_month,
        base_charge=base_charge,
        usage_charges=usage_charges,
        total_charge=total_charge,
        payment_status=payment_status,
        payment_date=payment_date,
    ))

billing_df = spark.createDataFrame(invoices)
billing_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.billing_invoices")
print(f"✔ Created {CATALOG_SCHEMA}.billing_invoices ({billing_df.count()} rows)")

print("\n--- Subscribers (sample) ---")
display(subscribers_df.limit(5))
print("\n--- Towers (sample) ---")
display(towers_df.limit(5))
print("\n--- Call Records (sample) ---")
display(records_df.limit(5))
print("\n--- Billing Invoices (sample) ---")
display(billing_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.telecom.subscribers", {
    "subscriber_id":       "Unique identifier for the subscriber",
    "subscriber_name":     "Full name of the subscriber (generated via Faker)",
    "phone_number":        "Phone number (generated via Faker)",
    "plan_name":           "Subscription plan: Basic, Standard, Premium, Unlimited, or Family",
    "contract_type":       "Contract duration: Monthly, 12-Month, 24-Month, or Prepaid",
    "monthly_charge":      "Monthly plan charge in USD (fixed per plan)",
    "data_limit_gb":       "Monthly data allowance in GB. -1 indicates unlimited data",
    "device_type":         "Primary device model",
    "region":              "Geographic service region (8 US regions)",
    "status":              "Account status: Active, Suspended, Churned, or Pending Activation",
    "activation_date":     "Date the subscription was activated (DateType)",
    "churn_date":          "Date the subscriber churned (DateType). NULL if not churned. Temporal event for survival analysis",
    "credit_score":        "Subscriber credit score (normal distribution, mean 700, range 300-850)",
    "is_5g_enabled":       "Whether the subscriber has 5G network access",
    "avg_monthly_data_gb": "Average monthly data consumption in GB. Correlated with plan tier",
})

apply_comments(f"{CATALOG}.telecom.call_records", {
    "record_id":           "Unique identifier for the usage record",
    "subscriber_id":       "Foreign key referencing subscribers.subscriber_id",
    "record_date":         "Timestamp of the call or session (TimestampType)",
    "record_type":         "Type of usage: Voice Call, SMS, Data Session, MMS, or Voicemail",
    "duration_minutes":    "Duration in minutes. 0.0 for SMS/MMS. Log-normal for voice calls and data sessions",
    "data_usage_mb":       "Data consumed in megabytes (log-normal for Data Sessions)",
    "charge_amount":       "Additional charge beyond plan. 0.0 if included in plan",
    "network_type":        "Network used: 5G, 4G LTE, 3G, or Wi-Fi Calling",
    "call_result":         "Call outcome: Completed, Missed, Dropped, Busy, or Voicemail",
    "tower_id":            "Region-correlated cell tower identifier (TWR-XX-XXX format)",
    "roaming_country":     "Country name if roaming; NULL for domestic usage",
    "is_international":    "Whether the record involved international roaming",
    "latency_ms":          "Network latency in milliseconds. 5G: ~12ms, 4G: ~30ms, 3G: ~80ms, WiFi: ~20ms",
    "signal_strength_dbm": "Signal strength in dBm. Higher (less negative) is better. Correlated with network type",
})

apply_comments(f"{CATALOG}.telecom.towers", {
    "tower_id":              "Unique cell tower identifier (TWR-XX-NNN format)",
    "region":                "Service region (matches subscribers.region)",
    "latitude":              "Approximate tower latitude (degrees N, US coverage band)",
    "longitude":             "Approximate tower longitude (degrees W, US coverage band)",
    "technology":            "RAN technology deployed at site: 5G, 4G LTE, 3G, or Wi-Fi backhaul",
    "capacity_subscribers":  "Estimated subscriber capacity (log-normal ~500–5000)",
    "sector_count":          "Number of antenna sectors (1–6)",
})

apply_comments(f"{CATALOG}.telecom.billing_invoices", {
    "invoice_id":      "Sequential unique invoice identifier",
    "subscriber_id":   "Foreign key referencing subscribers.subscriber_id",
    "billing_month":   "First day of the billed calendar month (DateType)",
    "base_charge":     "Monthly plan charge in USD (from plan monthly_charge)",
    "usage_charges":   "Overage and add-on usage charges. 0 for unlimited data plans",
    "total_charge":    "base_charge + usage_charges",
    "payment_status":  "Paid, Pending, Overdue, or Failed (weighted distribution)",
    "payment_date":    "Payment settlement date when Paid; NULL for Pending, Overdue, or Failed",
})

print(f"\n\u2705 All column comments applied for telecom schema")

spark.sql(f"ALTER TABLE {CATALOG}.telecom.subscribers ALTER COLUMN subscriber_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.subscribers ADD CONSTRAINT pk_subscribers PRIMARY KEY (subscriber_id)")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.call_records ALTER COLUMN record_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.call_records ADD CONSTRAINT pk_call_records PRIMARY KEY (record_id)")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.call_records ADD CONSTRAINT fk_call_records_subscriber_id FOREIGN KEY (subscriber_id) REFERENCES {CATALOG}.telecom.subscribers(subscriber_id)")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.towers ALTER COLUMN tower_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.towers ADD CONSTRAINT pk_towers PRIMARY KEY (tower_id)")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.billing_invoices ALTER COLUMN invoice_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.billing_invoices ALTER COLUMN subscriber_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.billing_invoices ADD CONSTRAINT pk_billing_invoices PRIMARY KEY (invoice_id)")
spark.sql(f"ALTER TABLE {CATALOG}.telecom.billing_invoices ADD CONSTRAINT fk_billing_invoices_subscriber_id FOREIGN KEY (subscriber_id) REFERENCES {CATALOG}.telecom.subscribers(subscriber_id)")
print(f"\u2714 PK/FK constraints applied for telecom schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.telecom') IS
'Telecom sample dataset with realistic statistical distributions and Faker-generated PII. Entity tables: `subscribers` (~5K rows, PK: subscriber_id), `towers` (~200 rows, PK: tower_id, region-correlated sites). Event/fact tables: `call_records` (100K-500K rows, PK: record_id, FK: subscriber_id → subscribers), `billing_invoices` (~30K–60K rows, PK: invoice_id, FK: subscriber_id → subscribers). Key features: Temporal churn dates (no call activity after churn), network quality metrics (latency/signal), tower dimension with geo/tech/capacity, monthly billing with payment status.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`telecom` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to telecom schema ({remove_after_value})")